# Week 3 Day 4 Lab 2 (free route) - UnderCover on Render

## Running the agent beside the app

On the AWS route, Lab 2 deploys the agent on AgentCore Runtime, where each dispatch runs in a session of its own, and points the app on Lambda at it. On the free route we run the agent as a second web service on Render, beside your app from Day 2, with the context layer inside the same service.

AgentCore Runtime, the Gateway and the Memory are AWS services, so for those you can watch me. The rest of today's build works the same way: the agent is the same AgentCore app, the app starts its work and keeps one run at a time, and a senior handler can send a claim back to the agent with a note.

This notebook has no code to run. It gives you the steps, with links to Render's own documentation.

## 5 steps:

STEP 1: Update the app on Render

STEP 2: Create the agent's web service

STEP 3: Point the app at the agent

STEP 4: Send a claim back

STEP 5: Compare with the AWS route

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of today's second lab. The main lab, <a href="week3day4_lab2.ipynb">week3day4_lab2.ipynb</a>, runs on AWS. You need the Render service from Day 2, and today's build of the product, yours or <code>week3day4-end</code>, pushed to GitHub as on Day 2.<br/><br/>I do this week on AWS, so I'm not recording the Render steps. Render's website changes from time to time, so if a screen looks different from what's described here, follow the linked page in Render's documentation.<br/><br/>Step 4 works one claim and then works it again, which uses part of your 50 free requests for the day. A dispatch works the whole queue, which needs more requests than a day's allowance, so on the free route we work one claim.
            </span>
        </td>
    </tr>
</table>

## Step 1: Update the app on Render

Today's build changes the app: it keeps one run at a time itself, and a senior handler can send a claim back. So the app on Render needs today's code.

- **To deploy my build,** open your web service in the [Render Dashboard](https://dashboard.render.com), go to **Settings**, change the **Branch** to `week3day4-end`, and save. Then choose **Manual Deploy** and **Deploy latest commit**.
- **To deploy your own build,** push your branch to your own GitHub repository, as on Day 2, and choose **Manual Deploy** and **Deploy latest commit**.

The build and start commands and the environment variables stay as they were on Day 2. Step 3 adds one more.

Render's documentation: [Deploys](https://render.com/docs/deploys).

## Step 2: Create the agent's web service

In the [Render Dashboard](https://dashboard.render.com), choose **New**, then **Web Service**, and pick the same repository and branch as your app. Then fill in the form:

| Field | Value |
|---|---|
| Name | `undercover-agent` |
| Region | The same as your app |
| Language | Python 3 |
| Build Command | `uv sync --frozen --only-group runtime --only-group context` |
| Start Command | `uv run --no-sync python -m context & uv run --no-sync uvicorn agent.entrypoint:app --host 0.0.0.0 --port $PORT` |
| Instance Type | Free |

Under **Advanced**, set the **Health Check Path** to `/ping`.

What the commands do:

- `--only-group runtime` installs the packages the agent needs on a server, which the AWS route uses on Runtime, and `--only-group context` adds the context layer's.
- The start command runs the context layer in the background, at `http://127.0.0.1:8006/mcp` inside the service, where the agent looks for it by default, and the agent's AgentCore app on Render's port, as `npm run dev` serves it on your laptop. Only the agent is on the internet.

Under **Environment Variables**, add these lines from your `.env` file, and one more:

| Key | Value |
|---|---|
| `OPENROUTER_API_KEY` | From your `.env` |
| `UNDERCOVER_API_KEY` | From your `.env`, the same key as the app's |
| `DATABASE_URL` | The same session pooler address as in your `.env` |
| `UNDERCOVER_API` | Your app's address on Render, such as `https://undercover.onrender.com` |

Away from AgentCore Runtime, the agent refuses any request without the agent's key, so only your app can start its work. Choose **Create Web Service**, and when the deploy is live, `https://<your-agent>.onrender.com/ping` answers with `"status": "Healthy"`.

Render's documentation: [Web services](https://render.com/docs/web-services), [Environment variables and secrets](https://render.com/docs/configure-environment-variables), [Health checks](https://render.com/docs/health-checks).

## Step 3: Point the app at the agent

Open your app's web service, and under **Environment**, add:

| Key | Value |
|---|---|
| `UNDERCOVER_AGENT` | Your agent's address on Render, such as `https://undercover-agent.onrender.com` |

Choose **Save and deploy**. The app now sends the agent's work to the agent on Render, with the agent's key, as it does on your laptop.

A free service spins down after 15 minutes without a request, and the first request after that takes about a minute. The app waits only 10 seconds for the agent, so before you start any work, open the agent's `/ping` address in your browser and wait until it answers.

Render's documentation: [Free instances](https://render.com/docs/free).

## Step 4: Send a claim back

The three approvals over the limit in the queue are CLM-1003, CLM-1008 and CLM-1011. In a terminal in the `undercover` folder, reseed and start everything:

```
npm run seed
npm run dev
```

In a second terminal, work one of them with the agent on your laptop:

```
uv run -m agent CLM-1011
```

If the agent approves it, the claim waits for sign-off, with the claim graph paused and its place saved in Supabase. If it declines or refers the claim instead, try one of the other two. Then stop `npm run dev` with Ctrl+C. The rest happens on Render, and Supabase's free plan allows only 15 database connections at once.

Open your app on Render and sign in as Dana Whitfield. On the claim's page, send it back with a note saying what to look at again. On CLM-1011, for example, Meridian Air has already paid the customer $350 for the lost bag.

The app on Render asks the agent on Render to resume the claim. The agent loads the claim's thread from Supabase, where your laptop left it, reads the note and decides again. Refresh the claim's page to watch its steps.

## Step 5: Compare with the AWS route

On the AWS route, the same send back starts a new session on AgentCore Runtime, which loads the claim's thread from Supabase in the same way. The agent is the same AgentCore app on both. The difference is in how it runs: on Render it's one process that stays up between requests and sleeps after 15 minutes without one, while Runtime starts a session for each piece of work, in a microVM of its own, and ends it when the work is done. Watch Part 5 of the main lab for a dispatch of 25 claim agents in one session.

There's nothing to tear down. Both free web services cost nothing, so keep them for Day 5. You can delete them from the Render Dashboard whenever you no longer need them.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: the app keeps one run at a time in the database, and hears from the agent every minute while it works. Stop the agent from the Render Dashboard partway through a dispatch or an evaluation, start it again, and see what the app says when you start new work, before and after five minutes.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>